# Seq2Seq Transformer 번역 과제

- 데이터셋: `shihyunlim/aihub-ko-en-everyday-expression`
- 태스크: 한국어(`ko`) -> 영어(`en`) Seq2Seq 번역
- 모델: Hugging Face 사전학습 모델이 아니라 PyTorch `nn.Transformer` layer 기반 직접 구현
- 학습: 각 하이퍼파라미터 설정당 1 epoch만 학습
- 평가: validation loss/perplexity로 하이퍼파라미터 탐색 후 best config를 test loss/perplexity 및 sample BLEU로 평가

이 노트북 하나만 실행하면 데이터 로드, 토크나이저 학습, Seq2Seq Transformer 학습, 탐색, 평가가 모두 진행됩니다.


In [27]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


## 1. 전체 구현 코드

PyTorch `nn.Transformer`를 사용해 encoder-decoder Seq2Seq 모델을 직접 정의합니다. 토크나이저는 한국어 OOV를 줄이기 위해 train split 일부에서 byte-level BPE vocabulary를 학습합니다.
BLEU는 외부 평가 패키지 없이 실습 환경에서 바로 실행할 수 있도록 간단한 smoothing sentence BLEU로 계산합니다.


In [28]:
import math
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass, asdict, replace
from typing import Iterable

import pandas as pd
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="The PyTorch API of nested tensors is in prototype stage.*")


SEED = 42
DATASET_NAME = "shihyunlim/aihub-ko-en-everyday-expression"
SRC_COL = "ko"
TGT_COL = "en"

PAD = "[PAD]"
UNK = "[UNK]"
BOS = "[BOS]"
EOS = "[EOS]"
SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3


@dataclass
class TrainConfig:
    name: str
    d_model: int
    nhead: int
    num_layers: int
    dim_feedforward: int
    dropout: float
    lr: float
    batch_size: int = 64
    epochs: int = 1


@dataclass
class DataConfig:
    train_samples: int = 50000
    final_train_samples: int = 150000
    valid_samples: int = 1000
    test_samples: int = 1000
    vocab_size: int = 16000
    max_len: int = 64
    num_workers: int = 0


def sample_configs(n_trials: int = 5, seed: int = SEED) -> list[TrainConfig]:
    rng = random.Random(seed)
    search_space = {
        "d_model": [192, 256, 320],
        "num_layers": [2, 3],
        "dim_feedforward": [512, 768, 1024],
        "dropout": [0.1, 0.2],
        "lr": [3e-4, 5e-4, 7e-4, 1e-3],
        "batch_size": [48, 64],
    }
    nhead_by_d_model = {
        192: [4, 6, 8],
        256: [4, 8],
        320: [8],
    }

    configs = []
    seen = set()
    while len(configs) < n_trials:
        d_model = rng.choice(search_space["d_model"])
        cfg_tuple = (
            d_model,
            rng.choice(nhead_by_d_model[d_model]),
            rng.choice(search_space["num_layers"]),
            rng.choice(search_space["dim_feedforward"]),
            rng.choice(search_space["dropout"]),
            rng.choice(search_space["lr"]),
            rng.choice(search_space["batch_size"]),
        )
        if cfg_tuple in seen:
            continue
        seen.add(cfg_tuple)
        d_model, nhead, num_layers, dim_feedforward, dropout, lr, batch_size = cfg_tuple
        configs.append(
            TrainConfig(
                name=(
                    f"random_{len(configs) + 1}_d{d_model}_h{nhead}_"
                    f"l{num_layers}_ff{dim_feedforward}_do{dropout}_lr{lr:g}_bs{batch_size}"
                ),
                d_model=d_model,
                nhead=nhead,
                num_layers=num_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                lr=lr,
                batch_size=batch_size,
                epochs=1,
            )
        )
    return configs


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def pick_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_splits(data_cfg: DataConfig):
    total = data_cfg.train_samples + data_cfg.valid_samples + data_cfg.test_samples
    raw = load_dataset(DATASET_NAME, split="train")
    raw = raw.shuffle(seed=SEED).select(range(total))
    train_end = data_cfg.train_samples
    valid_end = train_end + data_cfg.valid_samples
    return {
        "train": raw.select(range(0, train_end)),
        "valid": raw.select(range(train_end, valid_end)),
        "test": raw.select(range(valid_end, total)),
    }


def bilingual_text_iterator(dataset) -> Iterable[str]:
    for row in dataset:
        for column in (SRC_COL, TGT_COL):
            text = row[column]
            if text is not None:
                yield str(text)


def train_shared_tokenizer(dataset, vocab_size: int) -> Tokenizer:
    tokenizer = Tokenizer(BPE(unk_token=UNK))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()
    trainer = BpeTrainer(vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL_TOKENS)
    tokenizer.train_from_iterator(bilingual_text_iterator(dataset), trainer=trainer)
    assert tokenizer.token_to_id(PAD) == PAD_ID
    assert tokenizer.token_to_id(UNK) == UNK_ID
    assert tokenizer.token_to_id(BOS) == BOS_ID
    assert tokenizer.token_to_id(EOS) == EOS_ID
    return tokenizer


def encode(tokenizer: Tokenizer, text: str, max_len: int) -> list[int]:
    ids = tokenizer.encode(str(text)).ids[: max_len - 2]
    return [BOS_ID] + ids + [EOS_ID]


class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, src_tokenizer: Tokenizer, tgt_tokenizer: Tokenizer, max_len: int):
        self.data = hf_dataset
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        row = self.data[int(idx)]
        src = encode(self.src_tokenizer, row[SRC_COL], self.max_len)
        tgt = encode(self.tgt_tokenizer, row[TGT_COL], self.max_len)
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


def collate_batch(batch):
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD_ID)
    return src, tgt


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, : x.size(1)])


class TransformerSeq2Seq(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        max_len: int,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_ID)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_ID)
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=max_len + 8)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor) -> torch.Tensor:
        src_key_padding_mask = src.eq(PAD_ID)
        tgt_key_padding_mask = tgt_in.eq(PAD_ID)
        tgt_len = tgt_in.size(1)
        tgt_mask = torch.triu(
            torch.ones((tgt_len, tgt_len), device=tgt_in.device, dtype=torch.bool),
            diagonal=1,
        )

        src_emb = self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.positional_encoding(self.tgt_embedding(tgt_in) * math.sqrt(self.d_model))
        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.generator(out)


def make_loaders(splits, src_tokenizer, tgt_tokenizer, data_cfg: DataConfig, batch_size: int):
    datasets = {
        k: TranslationDataset(v, src_tokenizer, tgt_tokenizer, data_cfg.max_len)
        for k, v in splits.items()
    }
    return {
        "train": DataLoader(
            datasets["train"],
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_batch,
            num_workers=data_cfg.num_workers,
        ),
        "valid": DataLoader(
            datasets["valid"],
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_batch,
            num_workers=data_cfg.num_workers,
        ),
        "test": DataLoader(
            datasets["test"],
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_batch,
            num_workers=data_cfg.num_workers,
        ),
    }


def run_epoch(model, loader, criterion, optimizer, device, train: bool, scheduler=None):
    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    progress = tqdm(loader, leave=False, desc="train" if train else "eval")
    for src, tgt in progress:
        src = src.to(device)
        tgt = tgt.to(device)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        with torch.set_grad_enabled(train):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

        n_tokens = tgt_out.ne(PAD_ID).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
        progress.set_postfix(loss=total_loss / max(total_tokens, 1))
    return total_loss / max(total_tokens, 1)


def decode_ids(tokenizer: Tokenizer, ids: list[int]) -> str:
    filtered = []
    for idx in ids:
        if idx == EOS_ID:
            break
        if idx not in (PAD_ID, BOS_ID):
            filtered.append(idx)
    return tokenizer.decode(filtered)


def blocks_repeated_ngram(generated: torch.Tensor, candidate_id: int, ngram_size: int = 3) -> bool:
    tokens = generated.squeeze(0).tolist() + [candidate_id]
    if len(tokens) < ngram_size * 2:
        return False
    ngrams = [tuple(tokens[i : i + ngram_size]) for i in range(len(tokens) - ngram_size + 1)]
    return len(ngrams) != len(set(ngrams))


@torch.no_grad()
def translate(
    model,
    src_text: str,
    src_tokenizer: Tokenizer,
    tgt_tokenizer: Tokenizer,
    data_cfg: DataConfig,
    device,
    beam_size: int = 4,
    length_penalty: float = 0.7,
):
    model.eval()
    src = torch.tensor([encode(src_tokenizer, src_text, data_cfg.max_len)], dtype=torch.long, device=device)
    beams = [(torch.tensor([[BOS_ID]], dtype=torch.long, device=device), 0.0, False)]

    for _ in range(data_cfg.max_len - 1):
        candidates = []
        for generated, score, finished in beams:
            if finished:
                candidates.append((generated, score, True))
                continue

            logits = model(src, generated)[:, -1].squeeze(0)
            logits[[PAD_ID, UNK_ID, BOS_ID]] = -float("inf")
            if generated.size(1) < 4:
                logits[EOS_ID] = -float("inf")
            log_probs = torch.log_softmax(logits, dim=-1)
            top_scores, top_ids = torch.topk(log_probs, k=min(beam_size * 4, log_probs.numel()))

            added = 0
            for token_score, token_id in zip(top_scores.tolist(), top_ids.tolist()):
                token_id = int(token_id)
                if blocks_repeated_ngram(generated, token_id, ngram_size=3):
                    continue
                next_id = torch.tensor([[token_id]], dtype=torch.long, device=device)
                next_generated = torch.cat([generated, next_id], dim=1)
                candidates.append((next_generated, score + float(token_score), token_id == EOS_ID))
                added += 1
                if added >= beam_size:
                    break

        beams = sorted(
            candidates,
            key=lambda item: item[1] / (item[0].size(1) ** length_penalty),
            reverse=True,
        )[:beam_size]
        if all(finished for _, _, finished in beams):
            break

    best = max(beams, key=lambda item: item[1] / (item[0].size(1) ** length_penalty))[0]
    return decode_ids(tgt_tokenizer, best.squeeze(0).tolist())


def sentence_bleu(reference: str, hypothesis: str, max_n: int = 4) -> float:
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    if not hyp_tokens:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        ref_counts = Counter(tuple(ref_tokens[i : i + n]) for i in range(max(len(ref_tokens) - n + 1, 0)))
        hyp_counts = Counter(tuple(hyp_tokens[i : i + n]) for i in range(max(len(hyp_tokens) - n + 1, 0)))
        overlap = sum(min(count, ref_counts[gram]) for gram, count in hyp_counts.items())
        total = max(sum(hyp_counts.values()), 1)
        precisions.append((overlap + 1.0) / (total + 1.0))

    bp = 1.0 if len(hyp_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    return bp * math.exp(sum(math.log(p) for p in precisions) / max_n)


@torch.no_grad()
def evaluate_bleu(model, hf_dataset, src_tokenizer, tgt_tokenizer, data_cfg: DataConfig, device, n_examples: int = 100):
    scores = []
    examples = []
    for row in tqdm(hf_dataset.select(range(min(n_examples, len(hf_dataset)))), desc="bleu", leave=False):
        pred = translate(model, row[SRC_COL], src_tokenizer, tgt_tokenizer, data_cfg, device)
        ref = str(row[TGT_COL])
        scores.append(sentence_bleu(ref, pred))
        if len(examples) < 5:
            examples.append({"ko": row[SRC_COL], "reference_en": ref, "prediction_en": pred})
    return sum(scores) / max(len(scores), 1), examples


def train_one_config(cfg: TrainConfig, data_cfg: DataConfig, splits, src_tokenizer, tgt_tokenizer, device):
    loaders = make_loaders(splits, src_tokenizer, tgt_tokenizer, data_cfg, cfg.batch_size)
    model = TransformerSeq2Seq(
        src_vocab_size=src_tokenizer.get_vocab_size(),
        tgt_vocab_size=tgt_tokenizer.get_vocab_size(),
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        dropout=cfg.dropout,
        max_len=data_cfg.max_len,
    ).to(device)
    train_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
    eval_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), eps=1e-9)
    total_steps = max(len(loaders["train"]) * cfg.epochs, 1)
    warmup_steps = max(int(total_steps * 0.1), 1)

    def lr_lambda(step: int):
        step = step + 1
        if step <= warmup_steps:
            return step / warmup_steps
        return max((total_steps - step) / max(total_steps - warmup_steps, 1), 0.1)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    start = time.time()
    train_loss = None
    valid_loss = None
    for _ in range(cfg.epochs):
        train_loss = run_epoch(model, loaders["train"], train_criterion, optimizer, device, train=True, scheduler=scheduler)
        valid_loss = run_epoch(model, loaders["valid"], eval_criterion, None, device, train=False)

    return {
        **asdict(cfg),
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_ppl": math.exp(min(valid_loss, 20.0)),
        "elapsed_sec": round(time.time() - start, 2),
        "model": model,
    }


def run_experiment(data_cfg: DataConfig | None = None, configs: list[TrainConfig] | None = None):
    seed_everything()
    device = pick_device()
    data_cfg = data_cfg or DataConfig()
    configs = configs or sample_configs(n_trials=5, seed=SEED)

    print(f"device: {device}")
    print(f"dataset: {DATASET_NAME}")
    print(f"search data config: {data_cfg}")

    splits = load_splits(data_cfg)
    print({k: len(v) for k, v in splits.items()})
    print(splits["train"][0])

    shared_tokenizer = train_shared_tokenizer(splits["train"], data_cfg.vocab_size)
    print("shared vocab:", shared_tokenizer.get_vocab_size())

    results = []
    best = None
    for cfg in configs:
        print(f"\n=== search: {cfg.name} ===")
        result = train_one_config(cfg, data_cfg, splits, shared_tokenizer, shared_tokenizer, device)
        model = result.pop("model")
        results.append(result)
        if best is None or result["valid_loss"] < best["metrics"]["valid_loss"]:
            best = {"metrics": result, "model": model}

    results_df = pd.DataFrame(results).sort_values("valid_loss").reset_index(drop=True)
    print("\nHyperparameter search results")
    print(results_df)

    final_data_cfg = data_cfg
    final_splits = splits
    final_tokenizer = shared_tokenizer
    best_model = best["model"]
    final_metrics = best["metrics"]

    if data_cfg.final_train_samples > data_cfg.train_samples:
        print(f"\n=== final refit: {best['metrics']['name']} on {data_cfg.final_train_samples} samples ===")
        final_data_cfg = replace(data_cfg, train_samples=data_cfg.final_train_samples)
        final_splits = load_splits(final_data_cfg)
        final_tokenizer = train_shared_tokenizer(final_splits["train"], final_data_cfg.vocab_size)
        final_cfg = TrainConfig(**{k: best["metrics"][k] for k in TrainConfig.__dataclass_fields__})
        final_result = train_one_config(final_cfg, final_data_cfg, final_splits, final_tokenizer, final_tokenizer, device)
        best_model = final_result.pop("model")
        final_metrics = final_result

    test_loaders = make_loaders(final_splits, final_tokenizer, final_tokenizer, final_data_cfg, final_metrics["batch_size"])
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    test_loss = run_epoch(best_model, test_loaders["test"], criterion, None, device, train=False)
    bleu, examples = evaluate_bleu(best_model, final_splits["test"], final_tokenizer, final_tokenizer, final_data_cfg, device)

    print("\nBest search config")
    print(best["metrics"])
    print("\nFinal model metrics")
    print(final_metrics)
    print(f"test_loss: {test_loss:.4f}, test_ppl: {math.exp(min(test_loss, 20.0)):.2f}, sample_bleu: {bleu:.4f}")
    print(pd.DataFrame(examples))
    return {
        "results": results_df,
        "final_metrics": final_metrics,
        "test_loss": test_loss,
        "sample_bleu": bleu,
        "examples": examples,
        "best_model": best_model,
        "src_tokenizer": final_tokenizer,
        "tgt_tokenizer": final_tokenizer,
        "data_cfg": final_data_cfg,
    }


## 2. 실험 설정 및 랜덤 서치 후보 생성

과제 조건에 맞춰 각 설정은 `epochs=1`로 고정합니다. GPU 메모리가 부족하면 `batch_size`, `train_samples`, `final_train_samples`, `n_trials`를 줄이면 됩니다.
`sample_configs`는 재현 가능한 랜덤 서치입니다. `seed=42`를 고정해 같은 후보가 다시 생성되도록 했고, 각 후보는 validation loss로 비교합니다.
빠른 탐색은 5만 샘플에서 수행하고, validation loss가 가장 낮은 설정은 더 큰 15만 샘플에서 1 epoch만 final refit한 뒤 test 평가합니다.
한국어와 영어를 하나의 shared BPE vocabulary로 학습해 숫자, 고유명사, 부분 문자열을 source와 target에서 공유하도록 했습니다.


In [29]:
data_cfg = DataConfig(
    train_samples=50000,
    final_train_samples=150000,
    valid_samples=1000,
    test_samples=1000,
    vocab_size=16000,
    max_len=64,
)

configs = sample_configs(n_trials=5, seed=42)
pd.DataFrame([asdict(cfg) for cfg in configs])


,name,d_model,nhead,num_layers,dim_feedforward,dropout,lr,batch_size,epochs
0,random_1_d320_h8_l2_ff1024_do0.2_lr0.0005_bs48,320,8,2,1024,0.2,0.0005,48,1
1,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48,192,8,2,1024,0.1,0.0010,48,1
2,random_3_d192_h4_l2_ff512_do0.1_lr0.0005_bs64,192,4,2,512,0.1,0.0005,64,1
3,random_4_d192_h6_l3_ff512_do0.1_lr0.001_bs64,192,6,3,512,0.1,0.0010,64,1
4,random_5_d256_h4_l2_ff768_do0.1_lr0.0003_bs64,256,4,2,768,0.1,0.0003,64,1


## 3. 데이터셋 다운로드 후 학습, 탐색, 평가 실행

아래 셀은 Hugging Face에서 데이터셋을 다운로드합니다. 네트워크 연결이 필요합니다.


In [30]:
outputs = run_experiment(data_cfg=data_cfg, configs=configs)
outputs["results"]


device: cuda
dataset: shihyunlim/aihub-ko-en-everyday-expression
search data config: DataConfig(train_samples=50000, final_train_samples=150000, valid_samples=1000, test_samples=1000, vocab_size=16000, max_len=64, num_workers=0)
{'train': 50000, 'valid': 1000, 'test': 1000}
{'ko': '주문하신 것을 다시 불러드릴게요.', 'en': "I'll just repeat your order."}
shared vocab: 16000

=== search: random_1_d320_h8_l2_ff1024_do0.2_lr0.0005_bs48 ===


train:   0%|          | 0/1042 [00:00<?, ?it/s]

eval:   0%|          | 0/21 [00:00<?, ?it/s]


=== search: random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48 ===


train:   0%|          | 0/1042 [00:00<?, ?it/s]

...

eval:   0%|          | 0/21 [00:00<?, ?it/s]

eval:   0%|          | 0/21 [00:00<?, ?it/s]

bleu:   0%|          | 0/100 [00:00<?, ?it/s]


Best search config
{'name': 'random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48', 'd_model': 192, 'nhead': 8, 'num_layers': 2, 'dim_feedforward': 1024, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 48, 'epochs': 1, 'train_loss': 5.961890495814126, 'valid_loss': 4.800081578979297, 'valid_ppl': 121.52033061891481, 'elapsed_sec': 48.53}

Final model metrics
{'name': 'random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48', 'd_model': 192, 'nhead': 8, 'num_layers': 2, 'dim_feedforward': 1024, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 48, 'epochs': 1, 'train_loss': 5.506927244269606, 'valid_loss': 4.214955962443844, 'valid_ppl': 67.69118485093982, 'elapsed_sec': 143.97}
test_loss: 4.2220, test_ppl: 68.17, sample_bleu: 0.1370
                                                  ko  \
0  국제 배송이 도착하는 데 영업일 기준 2일이 소요되기 때문에 이에 대해 알려드리기 ...   
1              저는 한국에서 식물 제품을 판매하는 여러 지점을 소유하고 있습니다.   
2                        좋은 아침입니다, 오늘은 어디를 가고 싶으신가요?   
3                  교통사고와 교통법규 위반만 피하면 할인을 받을 수 있습니다.   
4    

,name,d_model,nhead,num_layers,dim_feedforward,dropout,lr,batch_size,epochs,train_loss,valid_loss,valid_ppl,elapsed_sec
0,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48,192,8,2,1024,0.1,0.0010,48,1,5.961890,4.800082,121.520331,48.53
1,random_4_d192_h6_l3_ff512_do0.1_lr0.001_bs64,192,6,3,512,0.1,0.0010,64,1,6.061623,4.907985,135.366422,50.27
2,random_1_d320_h8_l2_ff1024_do0.2_lr0.0005_bs48,320,8,2,1024,0.2,0.0005,48,1,6.130548,4.985599,146.291170,64.89
3,random_3_d192_h4_l2_ff512_do0.1_lr0.0005_bs64,192,4,2,512,0.1,0.0005,64,1,6.294323,5.159252,174.034146,42.97
4,random_5_d256_h4_l2_ff768_do0.1_lr0.0003_bs64,256,4,2,768,0.1,0.0003,64,1,6.329849,5.204501,182.089905,49.58


## 4. 번역 예시 확인


In [31]:
pd.DataFrame(outputs["examples"])


,ko,reference_en,prediction_en
0,국제 배송이 도착하는 데 영업일 기준 2일이 소요되기 때문에 이에 대해 알려드리기 ...,I'm calling ahead to inform you about this sin...,I'm going to know if you want to know how to m...
1,저는 한국에서 식물 제품을 판매하는 여러 지점을 소유하고 있습니다.,I own multiple stores in Korea that sell these...,I am looking for a new product.
2,"좋은 아침입니다, 오늘은 어디를 가고 싶으신가요?","Good morning, so where would you like to go to...","Good day, and how much do you want to buy?"
3,교통사고와 교통법규 위반만 피하면 할인을 받을 수 있습니다.,If you avoid car accidents and traffic violati...,"If you have any questions, you can be able to ..."
4,"네, 지난주에 예약금으로 30달러를 결제했어요.","Yes, I paid 30 dollars last week for the reser...","Yes, I'll send you a reservation for your comp..."


## 5. 결과 해석

- `valid_loss`가 가장 낮은 설정을 best config로 선택했습니다.
- `test_loss`, `test_ppl`, `sample_bleu`는 best config 모델로 test split에서 계산했습니다.
- `sample_bleu`는 외부 패키지 없이 구현한 간단한 sentence BLEU 평균이므로 공식 BLEU 점수와는 차이가 있을 수 있습니다.
- 이전 결과처럼 일반적인 영어 문구가 반복되는 현상은 1 epoch 학습에서 target 쪽 고빈도 표현을 과도하게 따라가고, source와 target vocab이 분리되어 입력 정보가 decoder까지 충분히 전달되지 않을 때 쉽게 발생합니다.
- 이를 줄이기 위해 shared byte-level BPE 토크나이저, 학습용 label smoothing, learning-rate warmup/decay, final refit, beam search, PAD/UNK/BOS 디코딩 차단, 3-gram 반복 방지 디코딩을 적용했습니다.
- 그래도 1 epoch만 학습했기 때문에 번역 품질은 제한적이며, 과제 조건상 모델 구조와 학습/평가 파이프라인 완성, 그리고 validation loss 기준 하이퍼파라미터 비교에 초점을 두었습니다.


## 6. 추가 실험: epoch 제한을 두지 않는 경우

아래 코드는 기존 과제 조건을 유지하되 `epochs=1` 제한만 완화한 추가 실험입니다.
- 데이터셋: 동일한 `shihyunlim/aihub-ko-en-everyday-expression`
- 모델: 동일한 PyTorch `nn.Transformer` 기반 Seq2Seq
- 토크나이저/평가/랜덤 서치 기준: 위 실험과 동일
- 차이점: best config를 여러 epoch 학습하고 validation loss 기준으로 best checkpoint를 선택

실행 시간이 길 수 있으므로 필요에 따라 `unlimited_data_cfg.train_samples`, `n_trials`, `unlimited_epochs`를 줄여도 됩니다.


In [33]:
def train_with_validation_history(
    cfg: TrainConfig,
    data_cfg: DataConfig,
    splits,
    tokenizer,
    device,
):
    loaders = make_loaders(splits, tokenizer, tokenizer, data_cfg, cfg.batch_size)
    model = TransformerSeq2Seq(
        src_vocab_size=tokenizer.get_vocab_size(),
        tgt_vocab_size=tokenizer.get_vocab_size(),
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        dropout=cfg.dropout,
        max_len=data_cfg.max_len,
    ).to(device)

    train_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
    eval_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), eps=1e-9)
    total_steps = max(len(loaders["train"]) * cfg.epochs, 1)
    warmup_steps = max(int(total_steps * 0.1), 1)

    def lr_lambda(step: int):
        step = step + 1
        if step <= warmup_steps:
            return step / warmup_steps
        return max((total_steps - step) / max(total_steps - warmup_steps, 1), 0.1)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    history = []
    best_state = None
    best_valid_loss = float("inf")
    start = time.time()

    for epoch in range(1, cfg.epochs + 1):
        print(f"\nEpoch {epoch}/{cfg.epochs}")
        train_loss = run_epoch(
            model,
            loaders["train"],
            train_criterion,
            optimizer,
            device,
            train=True,
            scheduler=scheduler,
        )
        valid_loss = run_epoch(model, loaders["valid"], eval_criterion, None, device, train=False)
        row = {
            **asdict(cfg),
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_ppl": math.exp(min(valid_loss, 20.0)),
            "elapsed_sec": round(time.time() - start, 2),
        }
        history.append(row)
        print({k: row[k] for k in ["epoch", "train_loss", "valid_loss", "valid_ppl", "elapsed_sec"]})

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


@torch.no_grad()
def evaluate_model_on_test(model, splits, tokenizer, data_cfg: DataConfig, cfg: TrainConfig, device):
    loaders = make_loaders(splits, tokenizer, tokenizer, data_cfg, cfg.batch_size)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    test_loss = run_epoch(model, loaders["test"], criterion, None, device, train=False)
    bleu, examples = evaluate_bleu(model, splits["test"], tokenizer, tokenizer, data_cfg, device)
    metrics = {
        "test_loss": test_loss,
        "test_ppl": math.exp(min(test_loss, 20.0)),
        "sample_bleu": bleu,
    }
    return metrics, examples


In [34]:
# 실행 시간이 긴 추가 실험입니다. 필요하면 값을 줄여서 실행하세요.
unlimited_epochs = 5
unlimited_data_cfg = DataConfig(
    train_samples=150000,
    final_train_samples=150000,
    valid_samples=1000,
    test_samples=1000,
    vocab_size=16000,
    max_len=64,
)

# 위 1-epoch 랜덤 서치 결과가 있으면 그 best config를 재사용하고,
# 없으면 같은 seed의 랜덤 후보 중 첫 번째 후보를 사용합니다.
if "outputs" in globals() and "results" in outputs:
    best_row = outputs["results"].iloc[0].to_dict()
    base_cfg = TrainConfig(**{k: best_row[k] for k in TrainConfig.__dataclass_fields__})
else:
    base_cfg = sample_configs(n_trials=5, seed=42)[0]

unlimited_cfg = replace(base_cfg, name=f"{base_cfg.name}_epochs{unlimited_epochs}", epochs=unlimited_epochs)
unlimited_cfg


TrainConfig(name='random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_epochs5', d_model=192, nhead=8, num_layers=2, dim_feedforward=1024, dropout=0.1, lr=0.001, batch_size=48, epochs=5)

In [35]:
seed_everything()
device = pick_device()
print(f"device: {device}")
print(f"unlimited data config: {unlimited_data_cfg}")
print(f"unlimited config: {unlimited_cfg}")

unlimited_splits = load_splits(unlimited_data_cfg)
unlimited_tokenizer = train_shared_tokenizer(unlimited_splits["train"], unlimited_data_cfg.vocab_size)
print("shared vocab:", unlimited_tokenizer.get_vocab_size())

unlimited_model, unlimited_history = train_with_validation_history(
    unlimited_cfg,
    unlimited_data_cfg,
    unlimited_splits,
    unlimited_tokenizer,
    device,
)

unlimited_history


device: cuda
unlimited data config: DataConfig(train_samples=150000, final_train_samples=150000, valid_samples=1000, test_samples=1000, vocab_size=16000, max_len=64, num_workers=0)
unlimited config: TrainConfig(name='random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_epochs5', d_model=192, nhead=8, num_layers=2, dim_feedforward=1024, dropout=0.1, lr=0.001, batch_size=48, epochs=5)
shared vocab: 16000

Epoch 1/5


train:   0%|          | 0/3125 [00:00<?, ?it/s]

eval:   0%|          | 0/21 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 5.745249677194706, 'valid_loss': 4.1969169914603155, 'valid_ppl': 66.4810531122455, 'elapsed_sec': 143.43}

Epoch 2/5


train:   0%|          | 0/3125 [00:00<?, ?it/s]

...

{'epoch': 4, 'train_loss': 4.267636902275753, 'valid_loss': 3.2588989788727254, 'valid_ppl': 26.02087183521897, 'elapsed_sec': 572.75}

Epoch 5/5


train:   0%|          | 0/3125 [00:00<?, ?it/s]

eval:   0%|          | 0/21 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 4.134725100235512, 'valid_loss': 3.178506209136169, 'valid_ppl': 24.01085954704653, 'elapsed_sec': 714.85}


,name,d_model,nhead,num_layers,dim_feedforward,dropout,lr,batch_size,epochs,epoch,train_loss,valid_loss,valid_ppl,elapsed_sec
0,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_...,192,8,2,1024,0.1,0.001,48,5,1,5.745250,4.196917,66.481053,143.43
1,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_...,192,8,2,1024,0.1,0.001,48,5,2,4.784956,3.692359,40.139431,285.79
2,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_...,192,8,2,1024,0.1,0.001,48,5,3,4.464787,3.415873,30.443502,429.75
3,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_...,192,8,2,1024,0.1,0.001,48,5,4,4.267637,3.258899,26.020872,572.75
4,random_2_d192_h8_l2_ff1024_do0.1_lr0.001_bs48_...,192,8,2,1024,0.1,0.001,48,5,5,4.134725,3.178506,24.010860,714.85


Epoch 제한을 완화하여 동일 best config를 5 epoch 학습한 결과, validation loss는 4.197에서 3.179로 감소했고 validation perplexity는 66.48에서 24.01로 감소했다. 이는 1 epoch 조건에서는 모델이 충분히 수렴하지 못했으며, 추가 학습을 통해 validation 성능이 지속적으로 개선됨을 보여준다. 5 epoch까지는 train loss와 valid loss가 함께 감소했으므로 뚜렷한 과적합은 관찰되지 않았다.


In [36]:
unlimited_metrics, unlimited_examples = evaluate_model_on_test(
    unlimited_model,
    unlimited_splits,
    unlimited_tokenizer,
    unlimited_data_cfg,
    unlimited_cfg,
    device,
)

print(unlimited_metrics)
pd.DataFrame(unlimited_examples)


eval:   0%|          | 0/21 [00:00<?, ?it/s]

bleu:   0%|          | 0/100 [00:00<?, ?it/s]

{'test_loss': 3.183095503958221, 'test_ppl': 24.121305701573448, 'sample_bleu': 0.1959988879135047}


,ko,reference_en,prediction_en
0,국제 배송이 도착하는 데 영업일 기준 2일이 소요되기 때문에 이에 대해 알려드리기 ...,I'm calling ahead to inform you about this sin...,I am calling to inform you that the internatio...
1,저는 한국에서 식물 제품을 판매하는 여러 지점을 소유하고 있습니다.,I own multiple stores in Korea that sell these...,I have several products in South Korea.
2,"좋은 아침입니다, 오늘은 어디를 가고 싶으신가요?","Good morning, so where would you like to go to...","Good morning, where do you want to go?"
3,교통사고와 교통법규 위반만 피하면 할인을 받을 수 있습니다.,If you avoid car accidents and traffic violati...,You can get a discount if you want to receive ...
4,"네, 지난주에 예약금으로 30달러를 결제했어요.","Yes, I paid 30 dollars last week for the reser...","Yes, the reservation fee for 30 dollars."


In [38]:
unlimited_examples

[{'ko': '국제 배송이 도착하는 데 영업일 기준 2일이 소요되기 때문에 이에 대해 알려드리기 위해 미리 전화를 드렸습니다.',
  'reference_en': "I'm calling ahead to inform you about this since international shipment takes a couple of business days to arrive.",
  'prediction_en': 'I am calling to inform you that the international delivery will arrive within two days.'},
 {'ko': '저는 한국에서 식물 제품을 판매하는 여러 지점을 소유하고 있습니다.',
  'reference_en': 'I own multiple stores in Korea that sell these plant products.',
  'prediction_en': 'I have several products in South Korea.'},
 {'ko': '좋은 아침입니다, 오늘은 어디를 가고 싶으신가요?',
  'reference_en': 'Good morning, so where would you like to go today?',
  'prediction_en': 'Good morning, where do you want to go?'},
 {'ko': '교통사고와 교통법규 위반만 피하면 할인을 받을 수 있습니다.',
  'reference_en': 'If you avoid car accidents and traffic violations, you can earn a discount.',
  'prediction_en': 'You can get a discount if you want to receive a discount and discount.'},
 {'ko': '네, 지난주에 예약금으로 30달러를 결제했어요.',
  'reference_en': 'Yes, I paid 30 do